# 04 · Train Hazards Detector (yolo_hazard.pt)

> **OWNER:** Member B (M4 · hazards + plates)
> **PREREQUISITES:** `03_prepare_hazards.ipynb` complete — `data/hazards_prepared/data.yaml`
> exists and cleared the 50-image-per-class minimum.
> **EXPECTED RUNTIME:** ~1 hour on a T4.
> **OUTPUTS:** `models/yolo_hazard.pt`, `models/yolo_hazard.onnx`, `models/MODEL_CARD_hazard.md`.

Same shape as `02_train_road_damage.ipynb`, adjusted for a dataset roughly
1/15th the size: a nano model, more epochs with more patience, and heavier
augmentation to squeeze usable signal out of ~300 images across 3 classes.

This model detects `ZEBRA_CROSSING` presence, `DAMAGED_SIGN`, and
`WATERLOGGING` as bounding boxes only. It does **not** score zebra-crossing
condition — that rubric exists in notebook 03 Step 1b for human annotation
and future downstream scoring, but nothing in this training run consumes it.
Do not present condition-scoring as a working feature of this model.

**Next notebook:** `05_prepare_plates.ipynb` (yours too — M4).

In [ ]:
# Cell 1/3 — minimal bootstrap (Colab vs local). No repo imports yet: on a
# fresh Colab runtime nothing has been cloned, so this cell is deliberately
# self-contained and only prepares sys.path so `common/` becomes importable.
import subprocess
import sys
from pathlib import Path


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()
REPO_URL = "https://github.com/ad8thya/SmartIndiaHackathon.git"

if IN_COLAB:
    REPO_ROOT = Path("/content/SmartIndiaHackathon")
    if not REPO_ROOT.exists():
        print(f"cloning {REPO_URL} -> {REPO_ROOT}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        print(f"{REPO_ROOT} already present locally on this runtime")
else:
    _here = Path.cwd().resolve()
    _candidates = [c for c in (_here, *_here.parents) if (c / "pyproject.toml").exists() and (c / "notebooks").exists()]
    if not _candidates:
        raise RuntimeError(
            "Could not find the repo root (looked for pyproject.toml + notebooks/ "
            f"walking up from {_here}). Run this notebook from inside the repo checkout."
        )
    REPO_ROOT = _candidates[0]

for _p in (str(REPO_ROOT), str(REPO_ROOT / "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Colab: {IN_COLAB}")
print(f"repo root: {REPO_ROOT}")

In [ ]:
# Cell 2/3 — install the ML extras. Quiet; ~60-90s on a fresh Colab runtime,
# near-instant if already installed (pip no-ops on a satisfied requirement).
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[ml]"],
    check=True,
)
print("ml extras installed")

In [ ]:
# Cell 3/3 — full environment setup (Drive mount + DATA_ROOT/MODEL_ROOT) and
# an import check: torch version + CUDA availability, so a broken environment
# fails here, not forty minutes into a training run.
from common import colab as colab_mod

env = colab_mod.setup_environment()
REPO_ROOT, DATA_ROOT, MODEL_ROOT = env["repo_root"], env["data_root"], env["model_root"]

import ultralytics

print(f"ultralytics {ultralytics.__version__}")
gpu_info = colab_mod.gpu_report()

## Step 1 — Assert data.yaml matches the frozen indices

In [ ]:
import yaml

from common import constants

DATA_YAML_PATH = DATA_ROOT / "hazards_prepared" / "data.yaml"
with open(DATA_YAML_PATH) as f:
    data_yaml = yaml.safe_load(f)

constants.assert_class_order(data_yaml["names"], constants.HAZARD_CLASSES, "hazard")

## Step 2 — Train

`yolo11n.pt` (nano — 400 images cannot support a larger model without
overfitting badly), epochs 100 with patience 25 (small data needs more passes
to converge, and more patience before early-stopping calls it). Heavier
augmentation than the RDD run: `mixup=0.15`, `hsv_v=0.5`, `scale=0.5`,
`degrees=5` — with this little data, augmentation is doing more of the work
that a larger dataset would otherwise do for free.

In [ ]:
import random

import numpy as np
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

RUNS_DIR = MODEL_ROOT / "runs" / "hazard"

hazard_model = YOLO("yolo11n.pt")
hazard_results = hazard_model.train(
    data=str(DATA_YAML_PATH),
    imgsz=640,
    epochs=100,
    batch=-1,
    optimizer="AdamW",
    lr0=1e-3,
    patience=25,
    seed=SEED,
    mosaic=1.0,
    mixup=0.15,
    hsv_v=0.5,
    scale=0.5,
    degrees=5,
    fliplr=0.5,
    project=str(RUNS_DIR),
    name="hazard",
    exist_ok=True,
)
print(f"run saved to {hazard_results.save_dir}")

## Step 3 — Evaluate on TEST + explicit overfitting check

With ~100 images/class, **val mAP will look better than reality** — the
validation set is small enough that the model can partially memorize it
across 100 epochs. Report the train-vs-val gap honestly; it goes in the model
card, not just this cell's output.

In [ ]:
from common import evaluate

test_table = evaluate.per_class_table(hazard_model, str(DATA_YAML_PATH), split="test")
print("TEST split:")
print(test_table.to_string(index=False))

val_table = evaluate.per_class_table(hazard_model, str(DATA_YAML_PATH), split="val")
train_map50 = float(hazard_results.results_dict.get("metrics/mAP50(B)", float("nan")))
val_map50 = float(val_table.loc[val_table["class"] == "ALL (mean)", "mAP50"].iloc[0])
test_map50 = float(test_table.loc[test_table["class"] == "ALL (mean)", "mAP50"].iloc[0])

gap = val_map50 - test_map50
print()
print(f"val mAP50:  {val_map50:.3f}")
print(f"test mAP50: {test_map50:.3f}")
print(f"val-test gap: {gap:+.3f}")
print()
print("WARNING: with ~100 images/class, this gap is expected to be non-trivial.")
print("A small val-test gap here is a property of the dataset size, not evidence")
print("the model generalizes well — report it honestly rather than quoting val alone.")

## Step 4 — Export + latency

In [ ]:
from common import export

HAZARD_PT_PATH = MODEL_ROOT / constants.MODEL_FILES["hazard"]
import shutil
from pathlib import Path

shutil.copy2(Path(hazard_model.trainer.best), HAZARD_PT_PATH)
onnx_path = export.export_onnx(HAZARD_PT_PATH, imgsz=640, opset=12)

latency = export.benchmark_latency(HAZARD_PT_PATH, onnx_path, imgsz=640)
export.print_latency_table(latency)

## Step 5 — Contact sheet + model card

Be honest in the card: this is a proof-of-concept classifier trained on a small self-annotated set, not a production model. That sentence goes in verbatim.

In [ ]:
from common import contact_sheet, model_card

sheet_path = contact_sheet.render_contact_sheet(
    images_dir=DATA_ROOT / "hazards_prepared" / "images" / "test",
    labels_dir=DATA_ROOT / "hazards_prepared" / "labels" / "test",
    class_names=constants.HAZARD_CLASSES,
    output_path=MODEL_ROOT / "hazard_contact_sheet_test.png",
    n=12,
)

model_card.render_model_card(
    model_name="yolo_hazard.pt — Hazards Detector",
    owner="M4",
    base_weights="yolo11n.pt",
    dataset="Self-annotated (Roboflow) — zebra crossings, damaged signs, waterlogging",
    dataset_size={"train": len(list((DATA_ROOT / "hazards_prepared" / "images" / "train").iterdir())),
                  "val": len(list((DATA_ROOT / "hazards_prepared" / "images" / "val").iterdir())),
                  "test": len(list((DATA_ROOT / "hazards_prepared" / "images" / "test").iterdir()))},
    class_names=constants.HAZARD_CLASSES,
    hyperparameters={
        "imgsz": 640, "epochs": 100, "optimizer": "AdamW", "lr0": 1e-3, "patience": 25,
        "seed": SEED, "mosaic": 1.0, "mixup": 0.15, "hsv_v": 0.5, "scale": 0.5, "degrees": 5, "fliplr": 0.5,
    },
    metrics_table_md=test_table.to_markdown(index=False) + f"\n\nval-test mAP50 gap: {gap:+.3f}",
    latency=latency,
    caveats=[
        "THIS IS A PROOF-OF-CONCEPT CLASSIFIER TRAINED ON A SMALL SELF-ANNOTATED SET, "
        "NOT A PRODUCTION MODEL. ~100 images/class is enough to demonstrate the pipeline "
        "end-to-end, not enough to claim generalized accuracy.",
        f"val-test mAP50 gap: {gap:+.3f} — val is expected to overstate real performance at this dataset size.",
        "Class balance and image sourcing were constrained by what was manually available "
        "in the sprint window; see notebook 03 for the per-class count table.",
        "DAMAGED_DIVIDER was dropped from this model (2026-08-28) — no dataset with a "
        "confirmed licence and a verifiably-labelled divider class could be sourced. The "
        "one Indian-specific lead (DATS_2022) could not be verified: its file API is "
        "inaccessible without a browser session, and its source paper never lists "
        "'divider' as one of its 45 annotated classes.",
        "WATERLOGGING's source imagery is MIXED/UNVERIFIED geography — a sample visibly "
        "includes both US content (highway route markers, US-style vehicles) and images "
        "that look South Asian in origin (clothing, demographics), consistent with a "
        "stock/news-photo aggregation. Per-image country is NOT confirmed either way — "
        "do not claim Indian-specific, but do not claim US-only either.",
        "FADED_ZEBRA was renamed ZEBRA_CROSSING (2026-08-28) — this model detects crossing "
        "PRESENCE only, any condition. It does NOT score how worn a crossing is.",
        "FUTURE WORK, NOT DEMONSTRATED: zebra-crossing condition scoring (repeated-pass "
        "comparison over the same road segment, or a heuristic on the detected crop) is "
        "designed and has a written annotation rubric (notebook 03 Step 1b), but no "
        "scoring code exists and none is run by this notebook. Describe it in the pitch "
        "as designed/future work — do not claim it is implemented or demonstrated.",
    ],
    output_path=MODEL_ROOT / "MODEL_CARD_hazard.md",
)

---
### What this notebook produced
- `models/yolo_hazard.pt`, `models/yolo_hazard.onnx`
- `models/MODEL_CARD_hazard.md` — honest about the proof-of-concept caveat

### Next
`05_prepare_plates.ipynb` (yours too — M4).